# CVPR Project 5 — KAT-style In-Context Imitation (ACT Transfer Cube)



## Pipeline 

| Step | What runs | Script |
|------|-----------|--------|
| 1 | Env smoke test | `sim_env` |
| 2 | Record 25 episodes + 4 subsets (`demos_{5,10,20}`, `test_5`) | `record_sim_episodes.py`, `scripts/make_subsets.py` |
| 3 | Tokenize OBS (DINO anchored keypoints + depth) + ACT (TRAJ/WP); decode-free replay sanity | `scripts/show_tokens.py`, `scripts/gt_replay.py` |
| 4 | Ollama ICL → JSON actions → replay in sim (single query smoke test) | `scripts/evaluate.py run --max_tests 1` + `scripts/replay_response.py` |
| 5 | Full seen / unseen eval + #demos, K, M ablations + aggregated tables | `scripts/evaluate.py sweep` |

## Seen vs. unseen

Episodes 0–24 are collected with a **fixed seed** (`--seed 0`), then split by episode index:

| Folder | Episodes | Role |
|--------|----------|------|
| `demos_5`, `demos_10`, `demos_20` | 0–4 / 0–9 / 0–19 | In-context demonstrations inserted in the prompt |
| `test_5` | **20–24** | **Held-out** queries — never present in any demo folder |

* **Seen** = leave-one-out queries on a demo folder (query episode IS in the demos but excluded from its own prompt).
* **Unseen / shifted** = queries on `test_5`, with a fresh random box pose drawn from the same simulator distribution.

Both rates are reported per CSV via the `split` column.

## How to use this notebook

**Prereqs**:

```bash
uv venv .venv --python 3.9
source .venv/bin/activate
uv pip install -r requirements-kat.txt   # single consolidated requirements file
cd detr && pip install -e . && cd ..
python3 scripts/download_dino_weights.py

ollama serve && ollama pull gemma4:26b && ollama pull llama3.2:latest
```

Every cell below shells out via `source .venv/bin/activate && python3 …` so the **uv-managed** venv at `./.venv` is the interpreter used for sim, DINO, and Ollama calls. Equivalent one-shot CLI (no notebook):

```bash
source .venv/bin/activate
python3 record_sim_episodes.py --task_name sim_transfer_cube_scripted \
    --dataset_dir data/transfer_cube/base --num_episodes 25 \
    --downsample_rate 10 --seed 0
python3 scripts/make_subsets.py --base_dir data/transfer_cube/base
python3 scripts/evaluate.py sweep --collection_seed 0 \
    --models 'gemma4:26b,llama3.2:latest'
```

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'act_kat').is_dir():
    raise SystemExit('Run notebook from repo root (folder containing act_kat/).')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL', 'gemma4:26b')
COLLECTION_SEED = int(os.environ.get('COLLECTION_SEED', '0'))
DATA_BASE = REPO_ROOT / 'data/transfer_cube/base'
SUBSETS  = REPO_ROOT / 'data/transfer_cube/subsets'
ARTIFACTS = REPO_ROOT / 'artifacts'
for sub in ('prompts', 'responses', 'replays'):
    (ARTIFACTS / sub).mkdir(parents=True, exist_ok=True)

def run(cmd: str, timeout_s: int = 7200):
    print(f'$ {cmd}\n')
    subprocess.run(cmd, shell=True, check=True, cwd=str(REPO_ROOT), timeout=timeout_s)

print('python    :', sys.executable)
print('repo root :', REPO_ROOT)
print('ollama    :', OLLAMA_MODEL)
print('data seed :', COLLECTION_SEED)

python    : /Users/sushantsubedi/Documents/act/.venv/bin/python
repo root : /Users/sushantsubedi/Documents/act
ollama    : gemma4:26b
data seed : 0


## Step 1 — Environment smoke test

In [2]:
run(r'''source .venv/bin/activate && python3 - <<'PY'
from sim_env import make_sim_env, BOX_POSE
from utils import sample_box_pose
BOX_POSE[0] = sample_box_pose()
env = make_sim_env('sim_transfer_cube_scripted')
ts = env.reset()
img = ts.observation['images']['top']
print('transfer_cube OK', img.shape, 'max_reward', env.task.max_reward)
PY''')

$ source .venv/bin/activate && python3 - <<'PY'
from sim_env import make_sim_env, BOX_POSE
from utils import sample_box_pose
BOX_POSE[0] = sample_box_pose()
env = make_sim_env('sim_transfer_cube_scripted')
ts = env.reset()
img = ts.observation['images']['top']
print('transfer_cube OK', img.shape, 'max_reward', env.task.max_reward)
PY

transfer_cube OK (480, 640, 3) max_reward 4


## Step 2 — Dataset (25 episodes, fixed seed) + subsets

In [ ]:
if not DATA_BASE.is_dir() or len(list(DATA_BASE.glob('episode_*.hdf5'))) < 25:
    run(f'''source .venv/bin/activate && mkdir -p {DATA_BASE} && \
python3 record_sim_episodes.py --task_name sim_transfer_cube_scripted \
  --dataset_dir {DATA_BASE} --num_episodes 25 --downsample_rate 10 \
  --seed {COLLECTION_SEED}''')
else:
    print(f'[skip] {DATA_BASE} already has >=25 episodes')

run(f'source .venv/bin/activate && python3 scripts/make_subsets.py --base_dir {DATA_BASE} --out_root {SUBSETS}')
run(f'source .venv/bin/activate && python3 visualize_episodes.py --dataset_dir {DATA_BASE} --episode_idx 0')

$ source .venv/bin/activate && mkdir -p /Users/sushantsubedi/Documents/act/data/transfer_cube/base && python3 record_sim_episodes.py --task_name sim_transfer_cube_scripted   --dataset_dir /Users/sushantsubedi/Documents/act/data/transfer_cube/base --num_episodes 25 --downsample_rate 10   --seed 0

episode_idx=0
Rollout out EE space scripted policy
episode_idx=0 Successful, episode_return=614
Replaying joint commands
episode_idx=0 Successful, episode_return=643
Saving: 0.0 secs

episode_idx=1
Rollout out EE space scripted policy
episode_idx=1 Successful, episode_return=612
Replaying joint commands
episode_idx=1 Successful, episode_return=638
Saving: 0.0 secs

episode_idx=2
Rollout out EE space scripted policy
episode_idx=2 Successful, episode_return=616
Replaying joint commands
episode_idx=2 Successful, episode_return=633
Saving: 0.0 secs

episode_idx=3
Rollout out EE space scripted policy
episode_idx=3 Successful, episode_return=622
Replaying joint commands
episode_idx=3 Successful, epi

## Step 3 — KAT tokenization + decode-free sanity

DINO ViT-S/16 → K best-buddy anchored patches → quantize (x,y,d) → `OBS_START / KP i x y d / OBS_END`.
Actions → M waypoints → 14-D per-dim quantizer → `TRAJ_START / WP[i=...] L=... LG=... R=... RG=... / TRAJ_END`.

In [ ]:
run('source .venv/bin/activate && python3 scripts/download_dino_weights.py')
run('source .venv/bin/activate && python3 scripts/show_tokens.py '
    '--episode_hdf5 data/transfer_cube/subsets/demos_5/episode_0.hdf5 '
    '--K 10 --M 20 --bins 32 --obs_bins 16 '
    '--overlay artifacts/keypoints_overlay_ep0_k10.png')

In [ ]:
run('source .venv/bin/activate && python3 scripts/gt_replay.py --dataset_dir data/transfer_cube/subsets/demos_5')
run('source .venv/bin/activate && python3 scripts/gt_replay.py --dataset_dir data/transfer_cube/subsets/test_5')

## Step 4 — Ollama ICL smoke test (single query)

Uses `evaluate.py run --max_tests 1` so it produces exactly one prompt/response and replays it.

In [ ]:
run(f'''source .venv/bin/activate && python3 scripts/evaluate.py run \
  --demos_dir data/transfer_cube/subsets/demos_5 \
  --tests_dir data/transfer_cube/subsets/test_5 \
  --eval_split unseen --max_tests 1 \
  --K 10 --M 20 --bins 32 --obs_bins 16 \
  --model {OLLAMA_MODEL} --collection_seed {COLLECTION_SEED} \
  --out_csv artifacts/results_smoke_unseen.csv''')

## Step 5 — Full sweep: seen + unseen + ablations + aggregate (one command)

Produces, under `artifacts/`:

* `results_p5_main_seen_unseen.csv` — main seen/unseen on `demos_20`
* `results_ablation_ndemos_{5,10,20}_unseen.csv` — #demos ablation
* `results_ablation_K{5,10,20}_unseen.csv` — keypoint count ablation
* `results_ablation_M{10,20,40}_unseen.csv` — action-token count ablation
* `results_ablation_model_{gemma4_26b,llama3_2_latest}_unseen.csv` — LLM model ablation (≈26B vs ≈3B)
* `p5_step5_summary.json`, `p5_step5_tables.md` — aggregated tables

Re-running with the same `--collection_seed` reproduces the same dataset; ICL queries pick the same demo / test episodes deterministically.

In [ ]:
MODELS = os.environ.get('OLLAMA_MODELS', 'gemma4:26b,llama3.2:latest')
run(f'source .venv/bin/activate && python3 scripts/evaluate.py sweep '
    f'--collection_seed {COLLECTION_SEED} --model {OLLAMA_MODEL} --models "{MODELS}"')

In [ ]:
p = ARTIFACTS / 'p5_step5_tables.md'
print(p.read_text() if p.is_file() else 'Run Step 5 first (needs Ollama).')

## Done

* Per-run logs live under `artifacts/prompts/`, `artifacts/responses/`, `artifacts/replays/`.
* Aggregated tables in `artifacts/p5_step5_tables.md`.
* Full write-up: `reports/P5_KAT_ICL_Report.md`.
* To replay any saved response on its test episode:

```bash
python3 scripts/replay_response.py \
  --response artifacts/responses/<run_id>.json \
  --test_episode_hdf5 data/transfer_cube/subsets/test_5/episode_20.hdf5 \
  --demos_dir data/transfer_cube/subsets/demos_5 --M 20
```